# LLM Fine-Tuning Deep Dive, Part 2 of 3: Where Should the Update Live?

> **The story:** Full fine-tuning was the original default because every weight remained available to learn. Freezing layers reduced optimizer state but still produced a complete changed model. LoRA made the update a small side path, and QLoRA asked one final question: if the base is frozen, why store it at full precision during training?
>
> **Where you are:** Part 1 showed that continued pretraining, SFT, and DPO teach different behaviors. Riverside can now choose the right learning objective, but its catalog changes and its hardware budget does not. This notebook keeps returning to one continued-pretraining goal while changing where the update is stored. Part 3 will decide what the resulting evidence can support.
>
> **Notation:** $W$ is a pretrained weight matrix; $\Delta W$ is its learned update; $A$ and $B$ are LoRA's low-rank matrices; $r$ is adapter rank; $\alpha$ scales the adapter path; $N$ is total parameters and $N_t$ is trainable parameters.

## 0 · The Challenge

> **The mission**: make Riverside's recurring catalog updates trainable and versionable on modest local hardware without changing the behavior objective under discussion.

**What we know so far:**

- Continued pretraining supplies a familiar next-token objective for Riverside prose.
- Full fine-tuning allows every weight to move.
- **But that freedom makes 100% of model parameters part of the trainable state and produces another complete checkpoint for each job.**

**What's blocking us:**
Riverside needs separate, versioned adaptations and must keep the base model available. Reducing trainable parameters helps only if the resulting artifact and resident-memory requirements also improve. Each strategy must therefore answer three practical questions: what learns, what stays resident, and what must be saved.

**What this chapter unlocks:**
A measured parameter-budget path: full fine-tuning as the reference, partial freezing as the first reduction, LoRA as the small swappable update, and QLoRA when the frozen base itself becomes the memory bottleneck. These are alternatives in production, not mandatory checkpoint ancestry; the order exists so each remaining cost motivates the next mechanism.

## Navigation

1. Reconstruct the common Riverside setup.
2. Measure full fine-tuning's trainable state.
3. Freeze layers and identify the cost that remains.
4. Build and inspect a LoRA adapter.
5. Introduce QLoRA only when frozen-base memory becomes the blocker.
6. Hand the candidates to [Part 3](03-llm-finetuning-comparison-and-decision.ipynb) for matched evaluation.

## The Budget Story: Remove One Bottleneck at a Time

Riverside keeps the learning objective fixed and changes only where trainable state lives and how large it is.

```mermaid
flowchart LR
    F["Full FT\nall weights learn"] -->|"all gradients and optimizer state"| P["Partial freezing\nselected layers learn"]
    P -->|"still saves a full changed model"| L["LoRA\nsmall adapter learns"]
    L -->|"base still occupies memory"| Q["QLoRA path\ncompact frozen base + adapter"]
    Q --> E["Part 3\nmatched quality and cost evidence"]
```

Read the arrows as discovery order, not a required production pipeline.

| Strategy | Cost it removes | Cost it leaves |
| --- | --- | --- |
| Partial freezing | Optimizer state for frozen layers | Manual layer choice and a complete changed model |
| LoRA | Full per-job weight updates and checkpoints | A full-precision frozen base in memory |
| QLoRA | Much of the frozen base's memory footprint | Low-bit kernel and hardware constraints |

This notebook measures parameter counts and artifact structure. It does not rank candidate quality from unmatched training runs; Part 3 performs that comparison.

![Parameter efficiency spectrum comparing full fine-tuning, partial fine-tuning, LoRA, and QLoRA](images/parameter-strategies-spectrum.png)

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(...).to(device)` loads pretrained weights and moves the model to the training device (CPU/GPU); `model.generate()` inside `torch.no_grad()` runs autoregressive decoding without tracking gradients. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(...)` loads weights, with `tf.device(...)` selecting the device; Keras has no built-in `.generate()` for causal LMs outside HF's `TFGenerationMixin.generate()`, and gradient tracking is simply skipped by not wrapping calls in a `tf.GradientTape()`.

In [ ]:
# Re-establish Part 1's lightweight TinyStories foundations.
from pathlib import Path
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "roneneldan/TinyStories-Instruct-8M"
SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
DEMO_TRAIN_STEPS = 20
STORY_SEED = "Aria Voss stared at the signal counting itself out in prime numbers and"
PROMPT = "Continue this fiction narrative in the same style: " + STORY_SEED

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model_config = base_model.config
print(
    f"Loaded {MODEL_NAME}: {len(base_model.transformer.h)} decoder layers, "
    f"hidden size {model_config.hidden_size}, "
    f"{sum(parameter.numel() for parameter in base_model.parameters()):,} parameters"
)


def format_instruction(prompt, tokenizer_obj=tokenizer):
    """Render the explicit TinyStories instruction contract used throughout the arc."""
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"### Instruction:\n{prompt.strip()}\n\n"
        "### Response:\n"
    )


def generate(model, prompt, max_new_tokens=60, instruction=True, tokenizer_obj=tokenizer):
    """Generate only new tokens using the shared instruction format when requested."""
    model.eval()
    model_input = format_instruction(prompt, tokenizer_obj) if instruction else prompt
    model_device = next(model.parameters()).device
    inputs = tokenizer_obj(model_input, return_tensors="pt")
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer_obj.pad_token_id,
            eos_token_id=tokenizer_obj.eos_token_id,
        )
    completion = tokenizer_obj.decode(
        output[0][prompt_len:], skip_special_tokens=True
    ).strip()
    return completion or "[model stopped immediately]"


print(f"Baseline completion (sanity check): {generate(base_model, PROMPT)}")

In [ ]:
# Corpus loader + the tokenize_causal() helper Concepts 5/6 reuse for training,
# plus every visualization/training/PEFT import this notebook needs.
try:

    # VS Code injects this variable so the notebook can locate its own folder
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:

        # Fall back to the script's own location when run outside VS Code's notebook runtime
        _notebook_dir = Path(__file__).parent
    except NameError:

        # Last resort: assume the current working directory is close enough
        _notebook_dir = Path.cwd()

CONTENT_DIR = _notebook_dir / "content"
if not CONTENT_DIR.exists():

    # Handle kernels whose cwd is the repo root instead of this notebook's own folder
    _fallback = Path.cwd() / "learning" / "genai" / "04-llm" / "content"
    if _fallback.exists():
        CONTENT_DIR = _fallback

print(f"Content directory: {CONTENT_DIR.absolute()}")

NOVELS = {
    "scifi": "the-weight-of-distant-light",
    "fantasy": "the-tidebound-accord",
    "mystery": "the-cartographers-cipher",
    "historical": "the-silk-merchants-daughter",
    "cyberpunk": "neural-drift",
    "horror": "the-hollow-beneath",
    "literary": "the-weight-of-tides",
}


def load_corpus_paragraphs(novels=None, min_len=200):
    """Load qualifying paragraphs from every chapter of the selected novels."""
    if novels is None:

        # Default to every novel in the catalog
        novels = list(NOVELS.keys())
    paragraphs = []
    for alias in novels:
        novel_dir = NOVELS.get(alias)
        if not novel_dir:

            # Skip aliases that don't map to a known novel
            continue
        novel_path = CONTENT_DIR / novel_dir
        if not novel_path.exists():

            # Skip novels whose content folder isn't present on disk
            continue

        # Read every chapter so training is not biased toward the beginnings of novels
        chapter_files = sorted(novel_path.glob("chapter-*.txt"))
        for path in chapter_files:
            text = path.read_text(encoding="utf-8")
            for para in text.split("\n\n"):

                # Collapse embedded newlines so each paragraph is a single line of text
                para = para.strip().replace("\n", " ")
                if len(para) >= min_len:

                    # Drop very short fragments that aren't real paragraphs
                    paragraphs.append(para)
    return paragraphs


def tokenize_causal(examples, tokenizer, max_length=128):
    """Tokenize one batched ``Dataset.map`` input for causal language modeling.

    Expected input shape: ``examples == {"text": list[str]}``, where
    ``len(examples["text"]) == B`` for the current batch. Because overflowing text is
    split into chunks, the returned ``input_ids``, ``attention_mask``, and ``labels``
    each have shape ``[B_chunks, max_length]``, where ``B_chunks`` may be greater than
    ``B``.
    """
    # Tokenize the batch, padding/truncating every example to the same length
    tokens = tokenizer(
        examples["text"], truncation=True, padding="max_length", max_length=max_length, return_overflowing_tokens=True
    )

    # Labels start as a copy of input_ids, with padding positions masked to -100
    labels = [
        [(tok if mask == 1 else -100) for tok, mask in zip(ids, attn)]
        for ids, attn in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    tokens["labels"] = labels
    return tokens


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Silence noisy library warnings that aren't actionable in this demo
warnings.filterwarnings("ignore")

# Apply a consistent figure resolution/font size to every plot in this notebook
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

# Apply a consistent seaborn theme/palette to every plot in this notebook
sns.set_theme(style="whitegrid", palette="muted")

print("Corpus loader, tokenize_causal(), and training/visualization imports ready.")

### Reloading Part 1's Instruction-Tuned LoRA Adapter

The later introspection section uses the instruction-tuning adapter produced by Part 1. The base model, tokenizer, target modules, and tensor shapes must match exactly.

> **Checkpoint migration boundary:** existing previous-model checkpoint folders are architecture-incompatible with TinyStories. This notebook does not modify them. Move or remove those old artifacts, then rerun Part 1 with `MODEL_NAME = "roneneldan/TinyStories-Instruct-8M"` to regenerate `./checkpoints/instruction-lora` and the other Part 1 artifacts before executing the reload below. Then rerun this notebook's training cells to regenerate `partial-freeze` and `peft-lora`.


> **PyTorch → Keras:** `PeftModel.from_pretrained(base_model, adapter_dir)` attaches previously trained LoRA adapter weights on top of a frozen base model, reconstructing the exact adapted model from Part 1's checkpoint. **Keras/TF equivalent:** there's no first-party Keras LoRA/PEFT library — the closest analog is reloading a full model (or a frozen-base-plus-trainable-sublayer subclass) via `model.load_weights(...)`; Keras users would typically just reload the entire fine-tuned model rather than a small swappable adapter.

In [ ]:
# Reload the instruction-tuned adapter only after Part 1 has regenerated it for MODEL_NAME.
instruction_adapter_dir = Path("./checkpoints/instruction-lora")
adapter_config_path = instruction_adapter_dir / "adapter_config.json"
if not adapter_config_path.is_file():
    raise FileNotFoundError(
        f"Missing {adapter_config_path}. Rerun Part 1 with MODEL_NAME={MODEL_NAME!r}."
    )

saved_adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
saved_base = saved_adapter_config.get("base_model_name_or_path")
if saved_base != MODEL_NAME:
    raise RuntimeError(
        f"Checkpoint base {saved_base!r} is incompatible with {MODEL_NAME!r}. "
        "Leave the old artifact untouched and regenerate Part 1 checkpoints."
    )

instruct_base_reload = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
instruct_lora_model = PeftModel.from_pretrained(
    instruct_base_reload, instruction_adapter_dir
).to(device)
instruct_lora_model.eval()
print("Reloaded the TinyStories instruction adapter generated by Part 1.")
print(generate(instruct_lora_model, PROMPT))


---

## Parameter Budget: Start with the Physical Bill

Part 1 chose the learning objective. Part 2 holds it steady and asks what training must keep in memory and what each job must save.

A trainable parameter needs a gradient and optimizer history. Freezing removes those two costs, but the weight and activations used by the forward pass remain.

### Concept 4: Full Fine-Tuning - The Unconstrained Reference

Every weight may change, so full fine-tuning is the reference for maximum update freedom and maximum trainable state. The next cell counts that state; it does not claim better Riverside behavior.

**Residual failure:** every adaptation trains and stores another complete model.

> **Bridge to partial freezing:** if early features are reusable, can Riverside stop paying optimizer state for all of them?

> **PyTorch → Keras:** `p.numel()` counts elements in each parameter tensor and `p.requires_grad` flags whether it receives gradient updates; summing over `model.parameters()` gives total vs. trainable parameter counts. **Keras/TF equivalent:** `model.count_params()` gives total parameters directly, and the trainable subset is `sum(np.prod(w.shape) for w in model.trainable_weights)` — Keras tracks trainable/non-trainable via each layer's `trainable` attribute rather than a per-tensor `requires_grad` flag.

In [ ]:
# Load a throwaway copy of the base model purely to count its parameters
param_check_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Every parameter tensor's element count contributes to the total
total = sum(p.numel() for p in param_check_model.parameters())

# With nothing frozen yet, every parameter is also trainable
trainable = sum(p.numel() for p in param_check_model.parameters() if p.requires_grad)
print(
    f"Full fine-tuning: {trainable:,}/{total:,} parameters trainable ({trainable / total * 100:.1f}%)"
)

# Free the memory now that the counts are captured
del param_check_model


### Concept 5: Partial Freezing - Stop Updating Reusable Layers

The base already represents general language. Riverside's first cost experiment freezes most transformer blocks and updates roughly the last quarter plus the final layer norm. GPT-Neo ties the token embedding to the language-model head, so both sides of that shared weight stay frozen. The block fraction is a testable hyperparameter, not a claim that domain knowledge always lives in late layers.

| What improves | What remains |
| --- | --- |
| Fewer gradients and optimizer slots | Full base participates in the forward pass |
| No adapter framework | Layer boundary is chosen manually |
| Frozen weights stay unchanged | Each adaptation remains a full checkpoint |

The next visualization verifies the physical split before discussing layer roles.

> **Bridge to LoRA:** can one shared frozen base serve many jobs while each job stores only its correction?

### Optional Visualization: Which Layers Are Trainable?

The policy is simple: freeze every parameter, then re-enable roughly the last quarter of `transformer.h` plus `transformer.ln_f`. The tied `transformer.wte`/`lm_head` weight stays frozen. The next figure makes that split visible; it does not claim that later blocks always contain domain knowledge or predict the gradients training will produce.

| Region | Policy in this experiment |
| --- | --- |
| Early and middle transformer blocks | Frozen |
| Last quarter of transformer blocks | Trainable |
| Final norm | Trainable |
| Tied token embedding / output head | Frozen |

Skip the figure if the parameter count is already sufficient. The code reports only the configured trainability mask and its measured parameter totals.

In [ ]:
# Optional visualization of the configured freezing policy; no gradient values are fabricated.
from transformers import AutoConfig
from matplotlib.patches import Patch

freeze_config = AutoConfig.from_pretrained(MODEL_NAME)
n_layers = freeze_config.num_layers
unfreeze_from = n_layers - max(2, n_layers // 4)
layers = [f"Block {index}" for index in range(n_layers)] + [
    "Final norm",
    "Tied embedding / output head",
]
trainable_mask = [index >= unfreeze_from for index in range(n_layers)] + [True, False]
colors = ["coral" if is_trainable else "lightblue" for is_trainable in trainable_mask]
positions = np.arange(len(layers))
tick_stride = max(1, n_layers // 12)
tick_positions = list(range(0, n_layers, tick_stride)) + [n_layers, n_layers + 1]

fig, axis = plt.subplots(figsize=(9, max(6, n_layers * 0.28)))
axis.barh(positions, np.ones(len(layers)), color=colors, edgecolor="black")
axis.set_yticks(tick_positions)
axis.set_yticklabels([layers[index] for index in tick_positions])
axis.set_xlim(0, 1)
axis.set_xticks([])
axis.invert_yaxis()
axis.set_title(
    f"Configured Partial-Freezing Policy ({n_layers} transformer blocks)",
    fontweight="bold",
)
axis.legend(
    handles=[
        Patch(facecolor="lightblue", edgecolor="black", label="Frozen"),
        Patch(facecolor="coral", edgecolor="black", label="Trainable"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.08),
    ncol=2,
)
plt.tight_layout()
plt.show()

frozen_blocks = unfreeze_from
trainable_blocks = n_layers - unfreeze_from
print(f"Frozen transformer blocks:    {frozen_blocks}/{n_layers}")
print(f"Trainable transformer blocks: {trainable_blocks}/{n_layers}")
print("Final norm:                   trainable")
print("Tied embedding/output head:   frozen")
print("This policy predicts trainable state, not gradient magnitude or model quality.")

> **PyTorch → Keras:** setting `param.requires_grad = False` on every parameter freezes the entire model so no gradients flow to it during backprop. **Keras/TF equivalent:** `layer.trainable = False` on each layer (or `model.trainable = False` for the whole model) before compiling — Keras freezes at layer granularity, not per-tensor, and the change only takes effect after the model is (re)compiled.

In [ ]:
# Load a fresh base model instance to selectively freeze/unfreeze
freeze_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)

# Start every parameter frozen; the next cell re-enables gradients on the trainable slice
for param in freeze_model.parameters():
    param.requires_grad = False


### Selectively Unfreezing the Last ~25% of Blocks

Every parameter starts frozen above. This cell re-enables `requires_grad` only on the last few
`transformer.h` blocks (`unfreeze_from` onward) plus `transformer.ln_f`, matching the split
visualized earlier. Because `lm_head.weight` is tied to `transformer.wte.weight`, both remain
frozen so the optimizer cannot silently update the token embedding through the output head.

> **PyTorch → Keras:** `model.named_parameters()` yields `(name, tensor)` pairs so individual parameters can be selectively re-enabled (`param.requires_grad = True`) by matching name substrings like block index or layer-norm/head names. **Keras/TF equivalent:** iterate `model.layers` and set `layer.trainable = True` for the specific layers you want to unfreeze (matched by `layer.name`), then recompile the model — Keras' selective-unfreezing story works the same way but at whole-layer granularity rather than per-parameter-tensor.

In [ ]:
n_layers = len(freeze_model.transformer.h)
unfreeze_from = n_layers - max(2, n_layers // 4)

# GPT-Neo exposes decoder blocks as transformer.h.
decoder_layers = freeze_model.transformer.h
for layer in decoder_layers[unfreeze_from:]:
    for parameter in layer.parameters():
        parameter.requires_grad = True
for parameter in freeze_model.transformer.ln_f.parameters():
    parameter.requires_grad = True

# lm_head is tied to transformer.wte. Leaving it frozen avoids silently unfreezing embeddings.
assert freeze_model.lm_head.weight is freeze_model.transformer.wte.weight
assert not freeze_model.transformer.wte.weight.requires_grad

trainable_layer_prefixes = tuple(
    f"transformer.h.{layer_index}." for layer_index in range(unfreeze_from, n_layers)
)
for name, parameter in freeze_model.named_parameters():
    if name.startswith(trainable_layer_prefixes) or name.startswith("transformer.ln_f."):
        assert parameter.requires_grad, f"Expected trainable GPT-Neo parameter: {name}"

trainable = sum(parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad)
total = sum(parameter.numel() for parameter in freeze_model.parameters())
print(
    f"Partial fine-tuning: {trainable:,}/{total:,} parameters trainable "
    f"({trainable / total * 100:.2f}%)"
)
print("Tied token embedding/output head remains frozen.")

### Building the Dataset and Running the Trainer

Same `tokenize_causal()`/`Trainer` pattern as continued pretraining, on a different 2-genre slice of
the corpus, with `learning_rate=1e-4` -- between full fine-tuning's `5e-5` and LoRA's `2e-4`, since
partial freezing updates more parameters than LoRA but far fewer than full fine-tuning.


> **PyTorch → Keras:** `Dataset`/`TrainingArguments`/`Trainer.train()` is HF's high-level PyTorch training loop (batching, optimizer, logging all handled internally), and `save_pretrained()` writes the model and config to disk. **Keras/TF equivalent:** `tf.data.Dataset` for batching, `model.compile(optimizer=..., loss=...)` to configure training, `model.fit(dataset, epochs=...)` to run it, and `model.save(...)` / `model.save_weights(...)` to persist — Keras' `fit()` plays the same role as `Trainer.train()`.

In [ ]:
# Partial freezing changes which parameters learn, not the causal-language-modeling objective.
# Load every chapter from two Riverside genres so later chapters are represented in training.

freeze_dataset = Dataset.from_dict(
    {"text": load_corpus_paragraphs(novels=["fantasy", "cyberpunk"])}
)

# Convert raw paragraphs into fixed-length input IDs, attention masks, and next-token labels.
# The same tokenize_causal() preprocessing is used for every parameter strategy so the
# comparison isolates the effect of freezing weights rather than changing the training data.

freeze_tokenized = freeze_dataset.map(
    lambda ex: tokenize_causal(ex, tokenizer), batched=True, remove_columns=["text"]
)

# Configure the optimization budget for the unfrozen parameter slice.

training_args_freeze = TrainingArguments(
    # Store this strategy separately because partial freezing produces a complete model checkpoint.

    output_dir="./checkpoints/partial-freeze",
    # Process two tokenized sequences in each forward/backward pass.

    per_device_train_batch_size=2,
    # Use the shared 20-update demonstration budget.

    max_steps=DEMO_TRAIN_STEPS,
    # Surface the training loss every ten steps so adaptation can be monitored.

    logging_steps=10,
    # Skip intermediate snapshots; the final trained model is saved explicitly below.

    save_strategy="no",
    # Use a middle-ground learning rate: higher than full fine-tuning, lower than LoRA,
    # because this strategy updates fewer parameters than full FT but many more than LoRA.

    learning_rate=1e-4,
    # Keep the run local instead of sending metrics to an external tracking service.

    report_to="none",
)

# Trainer receives the complete model, but its optimizer updates only tensors with
# requires_grad=True. Earlier cells enabled that flag only for the final ~25% of
# transformer.h and transformer.ln_f; the tied transformer.wte/lm_head remains frozen.

trainer_freeze = Trainer(
    model=freeze_model, args=training_args_freeze, train_dataset=freeze_tokenized
)

# Each step still runs a forward pass through every layer. During backpropagation, frozen
# layers retain their pretrained weights while gradients update only the unfrozen upper slice.

trainer_freeze.train()

# Unlike LoRA, partial freezing edits weights in the original architecture rather than storing
# a small adapter, so save the complete fine-tuned weights and config for later reloading.

freeze_model.save_pretrained("./checkpoints/partial-freeze")
print("Saved partial (layer-freezing) fine-tune checkpoint.")

### Concept 6: LoRA - Store the Correction, Not Another Base Model

Partial freezing still creates a complete changed checkpoint. LoRA instead learns a small correction beside a useful frozen transformation:

$$
\text{layer output}=Wx+B(Ax).
$$

$W$ is frozen; $BA$ is the learned correction. Rank $r$ is the bottleneck width and limits the independent directions that correction can express.

For one $256\times256$ TinyStories projection, full updating exposes $65{,}536$ weights. A rank-8 adapter uses $8\times256+256\times8=4{,}096$ trainable weights while reusing the original projection.

![A frozen 256-by-256 projection beside a rank-8 LoRA detour that compresses an activation to eight values, expands it back to 256, and adds the correction to the base output](images/lora-low-rank-adaptation.png)

The next cells attach rank-8 adapters and inspect the injected matrices. They show where the update lives and how much state is trainable, not whether LoRA matches full fine-tuning in quality.

**Residual failure:** the correction is small, but the frozen base still occupies ordinary model memory.

> **Bridge to QLoRA:** can Riverside store that frozen base more compactly?

> **PyTorch → Keras:** `LoraConfig(...)` declares the LoRA hyperparameters (rank, target modules, alpha), `get_peft_model(base, config)` wraps the frozen base model with trainable low-rank adapter matrices injected into the named target modules, and `print_trainable_parameters()` reports the resulting trainable/total ratio. **Keras/TF equivalent:** no first-party Keras LoRA API exists — the closest honest analog is manually freezing most layers (`layer.trainable = False`) and, for true low-rank adapters, hand-writing a custom `keras.layers.Layer` that adds a `BA` low-rank branch alongside a frozen dense layer; there's no drop-in `get_peft_model` equivalent.

In [ ]:
# GPT-Neo exposes separate query, key, value, and output projections in every attention block.
lora_config_pt = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

lora_pt_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
lora_pt_model = get_peft_model(lora_pt_base, lora_config_pt)
lora_pt_model.print_trainable_parameters()

### Building the Dataset

This run uses every chapter from three selected genres, following Part 1's policy for real training datasets. The novel list determines corpus breadth, while context length is set later by `tokenize_causal(max_length=128)`.

The selected genres differ from the partial-freezing run. That is acceptable for demonstrating the LoRA mechanics and measuring trainable parameters, but it does **not** form a controlled quality comparison between parameter strategies. Part 3 performs model selection under a shared evaluation context.

In [ ]:
# Load every qualifying paragraph from all chapters in the three selected genres.
# The novel list controls which domains enter this LoRA continued-pretraining run.
lora_pt_paragraphs = load_corpus_paragraphs(
    novels=["mystery", "horror", "literary"]
)

# Dataset.from_dict expects one column-oriented list: each paragraph becomes one raw row.
# This genre selection differs from the partial-freezing run, so it demonstrates LoRA's
# training mechanics; only parameter counts, not resulting model quality, are directly
# comparable across those runs.
lora_pt_dataset = Dataset.from_dict({"text": lora_pt_paragraphs})

# batched=True passes {"text": list[str]} into tokenize_causal(). Long paragraphs may
# produce multiple 128-token rows, while remove_columns drops the raw text after creating
# input_ids, attention_mask, and labels for causal next-token prediction.
lora_pt_tokenized = lora_pt_dataset.map(
    lambda examples: tokenize_causal(examples, tokenizer),
    batched=True,
    remove_columns=["text"],
)

# The later Trainer still stops after max_steps=DEMO_TRAIN_STEPS, so the shuffled run does not guarantee
# that one training pass visits every tokenized row in this full selected-novel corpus.
print(
    f"LoRA corpus: {len(lora_pt_paragraphs):,} paragraphs -> "
    f"{len(lora_pt_tokenized):,} fixed-length training chunks"
)

### Training and Saving

Same `2e-4` LoRA learning rate as instruction tuning, same `Trainer` pattern as every training cell in
this notebook -- this is the last of the five checkpoints this notebook trains before the head-to-head
comparison further down.


> **PyTorch → Keras:** same `TrainingArguments`/`Trainer.train()`/`save_pretrained()` pattern as the earlier training cell, here training only the LoRA adapter's parameters (the base stays frozen) and saving just the small adapter weights. **Keras/TF equivalent:** `model.compile(...)` + `model.fit(...)` with only the adapter sublayer's `trainable = True`, then `model.save_weights(...)` — since Keras has no adapter abstraction, this would typically save the whole model rather than a separate small adapter file.

In [ ]:
# Configure the training run: LoRA checkpoint dir, batch size, steps, higher LR than full FT
training_args_lora_pt = TrainingArguments(
    output_dir="./checkpoints/peft-lora",
    per_device_train_batch_size=2,
    max_steps=DEMO_TRAIN_STEPS,  # short instructional run; increase for a real convergence study
    logging_steps=10,
    save_strategy="no",
    learning_rate=2e-4,
    report_to="none",
)

# Wire the LoRA-wrapped model, its training args, and the tokenized dataset together
trainer_lora_pt = Trainer(
    model=lora_pt_model, args=training_args_lora_pt, train_dataset=lora_pt_tokenized
)

# Run training -- only the LoRA adapter matrices actually receive gradient updates
trainer_lora_pt.train()

# Persist just the small adapter weights (the frozen base isn't re-saved)
lora_pt_model.save_pretrained("./checkpoints/peft-lora")
print("Saved parameter-efficient (LoRA) continued-pretraining adapter.")


### Optional Depth: Visualize the Low-Rank Correction

The main practical path needs only the measured parameter count and a compatible adapter. The next visualization opens one $256\times256$ TinyStories projection and draws its rank-8 $A$ and $B$ path.

Use this section when you want to see why the correction has rank at most $r$. Otherwise, skip to **Crack Open the Adapter We Just Trained**.

For this projection, full updating exposes $65{,}536$ values; rank-8 LoRA exposes $4{,}096$, or 6.25%. The code derives these counts from the loaded model.

In [ ]:
# Optional shape-and-count view of one TinyStories projection.
from matplotlib.patches import FancyBboxPatch

projection_width = base_model.config.hidden_size
adapter_rank = 8
full_projection_params = projection_width * projection_width
adapter_a_params = adapter_rank * projection_width
adapter_b_params = projection_width * adapter_rank
adapter_total_params = adapter_a_params + adapter_b_params
adapter_percent = adapter_total_params / full_projection_params * 100

fig, (count_axis, path_axis) = plt.subplots(1, 2, figsize=(13, 4.5))

labels = ["Full projection\nupdate", "LoRA A + B"]
counts = [full_projection_params, adapter_total_params]
count_axis.bar(labels, counts, color=["steelblue", "mediumseagreen"], edgecolor="black")
count_axis.set_yscale("log")
count_axis.set_ylabel("Trainable values (log scale)")
count_axis.set_title("One Projection: Full Update vs LoRA", fontweight="bold")
for index, count in enumerate(counts):
    count_axis.text(index, count * 1.08, f"{count:,}", ha="center", va="bottom")

path_axis.set_xlim(0, 10)
path_axis.set_ylim(0, 3)
path_axis.axis("off")
boxes = [
    (0.3, 1.0, 1.5, 1.0, f"input x\n{projection_width} dims", "lightblue"),
    (2.4, 1.0, 1.7, 1.0, f"A\n{adapter_rank} x {projection_width}", "mediumseagreen"),
    (4.8, 1.0, 1.2, 1.0, f"rank-{adapter_rank}\nbottleneck", "#f6d365"),
    (6.7, 1.0, 1.7, 1.0, f"B\n{projection_width} x {adapter_rank}", "coral"),
    (9.0, 1.0, 0.7, 1.0, "delta", "#c7a4e0"),
]
for x_pos, y_pos, width, height, label, color in boxes:
    path_axis.add_patch(
        FancyBboxPatch(
            (x_pos, y_pos),
            width,
            height,
            boxstyle="round,pad=0.05",
            facecolor=color,
            edgecolor="black",
        )
    )
    path_axis.text(x_pos + width / 2, y_pos + height / 2, label, ha="center", va="center")
for start, end in [(1.8, 2.4), (4.1, 4.8), (6.0, 6.7), (8.4, 9.0)]:
    path_axis.annotate("", xy=(end, 1.5), xytext=(start, 1.5), arrowprops={"arrowstyle": "->"})
path_axis.set_title("The Adapter Path", fontweight="bold")

plt.tight_layout()
plt.show()

print(f"Full projection update: {full_projection_params:,} trainable values")
print(
    f"LoRA rank-{adapter_rank}: A={adapter_a_params:,} + B={adapter_b_params:,} "
    f"= {adapter_total_params:,} ({adapter_percent:.2f}% of the full projection)"
)
print("This comparison measures adapter capacity, not peak memory, speed, or quality.")

### Optional Deep Dive: Crack Open the Trained Adapter

The core LoRA idea is already established: a frozen projection plus a small trainable correction. The next cells verify that claim inside the actual Riverside adapter by locating PEFT's $A$ and $B$ matrices, checking the rank limit, and tracing one correction through a real projection.

Skip to **Adapter Portability** if the measured parameter count and saved adapter are enough for your goal.

> **PyTorch → Keras:** `model.named_modules()` walks the full module tree so code can find every submodule PEFT injected (checking for a `lora_A` attribute), then reads the frozen `base_layer` and trainable `lora_A`/`lora_B` matrices directly off the object. **Keras/TF equivalent:** `model.layers` (recursively via nested `submodules`) walks the layer graph, and each layer's `layer.weights`/`get_weights()` exposes its tensors — Keras has no PEFT-style wrapper object, so there's no `lora_A`/`lora_B` attribute to introspect unless you built the adapter yourself as a custom layer.

In [ ]:
# Find every TinyStories projection wrapped by the continued-pretraining LoRA adapter.
lora_layers = [
    (name, module)
    for name, module in lora_pt_model.named_modules()
    if hasattr(module, "lora_A") and len(getattr(module, "lora_A")) > 0
]
expected_adapters = len(lora_pt_model.base_model.model.transformer.h) * len(LORA_TARGET_MODULES)
print(
    f"PEFT wrapped {len(lora_layers)} projections; "
    f"expected {expected_adapters} for {len(LORA_TARGET_MODULES)} targets per decoder layer."
)
print("First wrapped module names:")
for name, _ in lora_layers[:8]:
    print(" ", name)

# Use the first query projection for concrete shape and rank introspection.
name0, layer0 = next(
    (name, module) for name, module in lora_layers if name.endswith("q_proj")
)
base0 = layer0.base_layer
lora_A0 = layer0.lora_A["default"]
lora_B0 = layer0.lora_B["default"]
scaling0 = layer0.scaling["default"]

print()
print(f"Inside {name0}:")
print(f"  Frozen q_proj:      {type(base0).__name__}, weight {tuple(base0.weight.shape)}")
print(f"  lora_A (down-proj): {tuple(lora_A0.weight.shape)}  <- trainable")
print(f"  lora_B (up-proj):   {tuple(lora_B0.weight.shape)}  <- trainable")
print(f"  scaling (alpha/r):  {scaling0}")
print(f"  Trained lora_B norm: {lora_B0.weight.norm().item():.4f}")

### Optional Check: Does the Update Really Have Rank at Most $r$?

Build PEFT's effective correction, $\Delta W=\text{scale}\cdot BA$, and inspect its singular values. Because $A$ passes through an $r$-wide bottleneck, no more than $r$ independent directions can remain non-zero. The next cell verifies that structural claim on the trained adapter.

> **PyTorch → Keras:** `layer.get_delta_weight(...)` (a PEFT method) computes the effective weight update `scaling · B @ A` as a plain tensor, and `torch.linalg.svdvals(...)` computes its singular values to verify the rank constraint. **Keras/TF equivalent:** `tf.linalg.svd(matrix, compute_uv=False)` computes singular values the same way; since Keras has no PEFT wrapper, you'd first need to manually multiply your own `B`/`A` weight matrices (`tf.matmul(B, A)`) to get the delta before taking its SVD.

In [ ]:
# Verify LoRA's rank constraint on the real trained adapter -- not asserted, measured via SVD.
delta_W = (
    layer0.get_delta_weight("default").detach().cpu()
)  # PEFT's own scaling * B @ A computation

# Singular values reveal how many independent directions ΔW actually has
singular_values = torch.linalg.svdvals(delta_W)
r = lora_A0.weight.shape[
    0
]  # the configured rank, read directly off the trained A matrix

# Only the first few dozen singular values are ever non-negligible for a rank-r update -- plotting
# all min(delta_W.shape) of them would squeeze the real cliff into an invisible sliver, so zoom in
# and use a log y-axis, which makes an 8-orders-of-magnitude drop actually visible.
n_show = min(30, len(singular_values))
floor = 1e-8  # log scale needs a positive floor; true near-zero values are clipped up for display only

# Clip near-zero singular values up to the floor so the log-scale axis can still plot them
plot_values = np.clip(singular_values[:n_show].numpy(), floor, None)

fig, ax = plt.subplots(figsize=(9, 4.5))

# Color the first r bars (real degrees of freedom) differently from the rest
ax.bar(
    range(n_show),
    plot_values,
    color=["mediumseagreen" if i < r else "lightgray" for i in range(n_show)],
)
ax.set_yscale("log")
ax.set_xlabel(f"Singular value index (first {n_show} of {len(singular_values)})")
ax.set_ylabel("Singular value magnitude (log scale)")
ax.set_title(
    f"Singular Value Spectrum of the Real Trained \u0394W = scaling \u00b7 B\u00b7A\n"
    f"({tuple(delta_W.shape)} matrix, configured rank r={r})",
    fontsize=11,
    fontweight="bold",
)
ax.legend(
    handles=[
        Patch(
            facecolor="mediumseagreen",
            edgecolor="black",
            label=f"First {r} singular values (the adapter's real degrees of freedom)",
        ),
        Patch(
            facecolor="lightgray",
            edgecolor="black",
            label=f"Indices {r + 1}-{n_show} (~0 by construction, floored for the log scale)",
        ),
    ],
    fontsize=8,
)
plt.tight_layout()
plt.show()

# Count how many singular values are meaningfully non-zero
nonzero = (singular_values > 1e-4).sum().item()
print(
    f"\u0394W shape: {tuple(delta_W.shape)} -> full rank would allow up to {min(delta_W.shape)} singular values"
)
print(f"Singular values above 1e-4: {nonzero} (matches the configured rank r={r})")
print(
    f"Largest singular value: {singular_values[0].item():.4f}   {r}th singular value: {singular_values[r - 1].item():.4f}"
)
print(
    f"First value past the rank cutoff (index {r}): {singular_values[r].item():.2e}  <- effectively zero"
)
print(
    "This is the real mechanics behind 'low-rank update': it isn't that training happened to find a "
    "low-rank \u0394W -- B(A(x))'s construction makes it mathematically impossible for \u0394W to have "
    "more than r independent directions, no matter what A and B learn."
)


### Optional Trace: See the Correction in One Forward Pass

Before training, PEFT initializes the adapter so it contributes no correction. After training, the combined projection differs slightly from the frozen base path. Forward hooks capture both outputs for one Riverside prompt, making the learned nudge visible without reimplementing PEFT internals.

> **PyTorch → Keras:** `module.register_forward_hook(...)` attaches a callback that captures a layer's output tensor during the forward pass, and the model call runs inside `torch.no_grad()` since no training is happening. **Keras/TF equivalent:** the closest analog is building an auxiliary `keras.Model` whose outputs include the intermediate layer(s) you want (`keras.Model(inputs=model.input, outputs=[layer.output, model.output])`), since Keras has no direct hook API; gradient tracking is simply avoided by not using a `tf.GradientTape()`.

In [ ]:
# Capture one real TinyStories query projection before and after its LoRA correction.
from matplotlib.patches import Patch

captured = {}


def make_hook(key):
    def hook(module, inputs, output):
        captured[key] = output.detach().cpu()
    return hook


hook_base = base0.register_forward_hook(make_hook("base_only"))
hook_combined = layer0.register_forward_hook(make_hook("combined"))
demo_prompt = STORY_SEED
enc_lora = tokenizer(demo_prompt, return_tensors="pt").to(device)
lora_pt_model.eval()
with torch.no_grad():
    _ = lora_pt_model(**enc_lora)
hook_base.remove()
hook_combined.remove()

base_out = captured["base_only"][0]
combined_out = captured["combined"][0]
lora_delta = combined_out - base_out
last_pos = base_out.shape[0] - 1
show_dims = min(60, base_out.shape[-1])

print(f"Raw continuation prompt: {demo_prompt!r}")
print(
    f"q_proj output shape: {tuple(base_out.shape)}; "
    "TinyStories uses separate q_proj/k_proj/v_proj/out_proj modules"
)
print()
print("At the last token position:")
print(f"  ||base q_proj output|| = {base_out[last_pos].norm().item():.3f}")
print(f"  ||LoRA delta||         = {lora_delta[last_pos].norm().item():.5f}")
print(
    "  delta / base norm     = "
    f"{(lora_delta[last_pos].norm() / base_out[last_pos].norm()).item():.4%}"
)

fig_static, (ax_static1, ax_static2) = plt.subplots(1, 2, figsize=(14, 4))
ax_static1.plot(
    base_out[last_pos, :show_dims].numpy(),
    color="steelblue",
    label="frozen q_proj",
)
ax_static1.plot(
    combined_out[last_pos, :show_dims].numpy(),
    color="coral",
    linestyle="--",
    label="q_proj + LoRA",
)
ax_static1.set_title(
    f"TinyStories Query Projection - first {show_dims} dimensions", fontsize=11
)
ax_static1.set_xlabel("q_proj output dimension")
ax_static1.legend(fontsize=8)
ax_static1.grid(alpha=0.3)

ax_static2.bar(
    np.arange(show_dims),
    lora_delta[last_pos, :show_dims].numpy(),
    color="mediumseagreen",
    label="LoRA delta",
)
ax_static2.set_title(
    f"LoRA delta at token position {last_pos}: scaling * B(A(x))", fontsize=11
)
ax_static2.set_xlabel("q_proj output dimension")
ax_static2.axhline(0, color="black", linewidth=0.8)
ax_static2.legend(fontsize=8)
ax_static2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

n_tokens = lora_delta.shape[0]
delta_arr = lora_delta[:, :show_dims].numpy()
delta_max = np.abs(delta_arr).max() * 1.15 or 1e-6
fig_anim, ax_anim = plt.subplots(figsize=(12, 4))
bar_colors = ["mediumseagreen" if value >= 0 else "coral" for value in delta_arr[0]]
bars_anim = ax_anim.bar(np.arange(show_dims), delta_arr[0], color=bar_colors)
ax_anim.axhline(0, color="black", linewidth=0.8)
ax_anim.set_ylim(-delta_max, delta_max)
ax_anim.set_xlim(-1, show_dims)
ax_anim.set_xlabel(f"q_proj output dimension (first {show_dims})")
ax_anim.set_ylabel("LoRA delta")
tokens_decoded = tokenizer.convert_ids_to_tokens(enc_lora["input_ids"][0].tolist())
title_obj = ax_anim.set_title("")


def _update(frame):
    deltas = delta_arr[frame]
    for bar, value in zip(bars_anim, deltas):
        bar.set_height(value)
        bar.set_color("mediumseagreen" if value >= 0 else "coral")
    title_obj.set_text(
        f"TinyStories q_proj LoRA delta - token {frame}/{n_tokens - 1} "
        f"{tokens_decoded[frame]!r}; ||delta|| = {np.linalg.norm(deltas):.5f}"
    )
    return list(bars_anim) + [title_obj]


anim = FuncAnimation(fig_anim, _update, frames=n_tokens, interval=160, blit=False)
plt.close(fig_anim)
display(HTML(anim.to_jshtml(fps=6)))

### Adapter Portability: Swapping Onto a Compatible Base

LoRA adapters are swappable because they store small deltas rather than another complete base model. Compatibility is stricter than matching one matrix width: the adapter must use the same base architecture, target-module names, tensor shapes, tokenizer contract, and preferably the exact base revision used for training.

A freshly instantiated `roneneldan/TinyStories-Instruct-8M` base is compatible with adapters regenerated by Parts 1 and 2 for that same identifier. A checkpoint produced for another architecture is not compatible, even when a few dimensions happen to match. PEFT validates the configured targets and tensor shapes during loading.

The code below loads the continued-pretraining adapter, swaps in Part 1's instruction adapter on the same resident TinyStories base, and inspects the actual query-projection adapter shape.


> **PyTorch → Keras:** `PeftModel.from_pretrained(...)` attaches one adapter to a base model, `load_adapter(...)` registers a second adapter on the same base without reloading its weights, and `set_adapter(...)` switches which adapter's deltas are active — all cheap pointer/dict operations inside PEFT's routing layer. **Keras/TF equivalent:** no built-in adapter-swapping mechanism exists; the practical substitute is keeping separate fully fine-tuned (or separately frozen/unfrozen) model copies and calling `model.load_weights(...)` to switch between them, which is far more expensive than PEFT's adapter swap since it reloads full weight sets rather than a tiny delta.

In [ ]:
import gc
import os
from peft import PeftModel

swap_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
swap_model = PeftModel.from_pretrained(swap_base, "./checkpoints/peft-lora")
swap_model.eval()

adapter_kb = sum(
    os.path.getsize(os.path.join("./checkpoints/peft-lora", filename))
    for filename in os.listdir("./checkpoints/peft-lora")
) / 1024
base_mb = sum(
    parameter.numel() * parameter.element_size()
    for parameter in swap_base.parameters()
) / 1024**2
print(f"Base model in memory: {base_mb:.0f} MB (shared across adapters)")
print(f"Continued-pretraining adapter on disk: {adapter_kb:.0f} KB")
print()
print("[Continued-pretraining adapter]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

swap_model.load_adapter("./checkpoints/instruction-lora", adapter_name="instruct")
swap_model.set_adapter("instruct")
print()
print("[Instruction adapter]")
print(f"  {generate(swap_model, PROMPT)}")

swap_model.set_adapter("default")
print()
print("[Continued-pretraining adapter, restored]")
print(f"  {generate(swap_model, STORY_SEED, instruction=False)}")

q_proj_adapter = next(
    module
    for name, module in swap_model.named_modules()
    if name.endswith("q_proj")
    and hasattr(module, "lora_A")
    and "default" in module.lora_A
)
q_proj_A_shape = tuple(q_proj_adapter.lora_A["default"].weight.shape)
print()
print(f"q_proj LoRA A shape: {q_proj_A_shape}")
print(
    f"Compatibility contract: base={MODEL_NAME}, "
    f"hidden_size={swap_model.config.hidden_size}, targets={LORA_TARGET_MODULES}"
)
print(
    "Adapters from the previous architecture must be regenerated; "
    "checkpoint files are not converted in place."
)

del swap_base, swap_model
gc.collect()


### Concept 7 (Parameter-Based): QLoRA - Put the Frozen Base on a Memory Diet

Ordinary LoRA solved one problem: very little state receives gradients or optimizer history. It did **not** make the frozen base disappear.

Return to Riverside's continued-pretraining adapter. The adapter may be tiny, yet every forward pass still needs the full base model in memory. For TinyStories that is manageable. For a multi-billion-parameter model, the frozen state can be the reason training does not fit on one GPU.

The next question is therefore:

> If the base will never be updated, can it be stored more compactly while the small LoRA path remains trainable?

That combination is **QLoRA**: keep the base frozen in a low-bit representation and train floating-point LoRA matrices beside it.

![One activation splitting into a frozen base path stored as four-bit codes with shared scales and reconstructed only for computation, and a trainable LoRA path whose correction rejoins the base output while gradients update only the adapter](images/qlora-quantized-base-lora-adapters.png)

Read the diagram as two paths through the same layer:

1. **Frozen base path:** compact codes and small scale values represent the base weights in memory. The runtime reconstructs the needed weight values into a compute-friendly type for matrix multiplication.
2. **Trainable adapter path:** LoRA $A$ and $B$ remain ordinary floating-point parameters, receive gradients, and produce the learned correction.
3. **Addition:** the base output and adapter correction are added exactly as in ordinary LoRA.

The common 4-bit format used for the frozen base is called **NF4**. Its implementation is blockwise: nearby weights share scale metadata, which lets small codes represent local ranges more accurately than one scale for the entire model. The important intuition is not the codebook formula; it is that compact storage and arithmetic precision are separate choices. The base can be stored in four-bit codes without asking the matrix multiplication itself to operate as crude four-bit arithmetic.

```mermaid
flowchart LR
    X["Activation"] --> BASE["Reconstruct frozen base weights<br/>for this computation"]
    CODES["Compact frozen codes<br/>+ scales"] --> BASE
    X --> A["Trainable LoRA A"] --> B["Trainable LoRA B"]
    BASE --> ADD(("Add"))
    B --> ADD
    ADD --> Y["Layer output"]
```

During backpropagation, gradients may pass through the base computation so earlier activations receive useful signals, but the frozen codes do not update. Optimizer state is needed only for the LoRA matrices.

### What the CPU Exercise Can and Cannot Show

The next cell uses a tiny uniform four-bit approximation because this notebook is designed to run without a CUDA-specific QLoRA stack. It demonstrates the structure:

- compact frozen codes are reconstructed for computation;
- a floating-point LoRA branch is added;
- gradients land only on the adapter.

It is **not** an NF4 implementation, a GPU memory benchmark, a trained QLoRA checkpoint, or evidence about QLoRA quality. A real run needs supported GPU kernels and libraries such as bitsandbytes, then the same matched quality evaluation required for every other parameter strategy.

TinyStories is small enough that ordinary LoRA is the honest choice for this local run. QLoRA becomes useful when the frozen base, not the adapter, is the memory bottleneck.

The deeper kernel and format details belong in [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb#appendix-b-nf4-and-qlora), after this storage-versus-compute intuition is secure.

In [ ]:
# CPU-only structural analogy: uniform 4-bit codes, not bitsandbytes NF4.
torch.manual_seed(7)
in_features, out_features, rank = 4, 3, 2
alpha = 4

activation = torch.randn(2, in_features)
base_weight = torch.randn(out_features, in_features)  # frozen reference weights

# Approximate each output row with signed 4-bit codes and one scale.
scale = base_weight.abs().amax(dim=1, keepdim=True).clamp_min(1e-8) / 7
quantized_codes = torch.clamp(torch.round(base_weight / scale), -8, 7).to(torch.int8)
reconstructed_weight = quantized_codes.float() * scale

# Only these small floating-point matrices are trainable.
lora_a = torch.nn.Parameter(torch.randn(rank, in_features) * 0.05)
lora_b = torch.nn.Parameter(torch.randn(out_features, rank) * 0.05)

base_output = activation @ reconstructed_weight.T
adapter_output = (activation @ lora_a.T) @ lora_b.T * (alpha / rank)
layer_output = base_output + adapter_output
layer_output.square().mean().backward()

print(f"Stored base codes: {quantized_codes.dtype}; reconstructed: {reconstructed_weight.dtype}")
print(f"Base is frozen: {base_weight.requires_grad is False}")
print(f"LoRA gradients: A={lora_a.grad.norm():.4f}, B={lora_b.grad.norm():.4f}")
print(f"Toy reconstruction MAE: {(base_weight - reconstructed_weight).abs().mean():.4f}")


### Compare Only What This Notebook Measured

Before changing a selected artifact for deployment, close the training-strategy story. The trained objects support two direct comparisons: how many parameters are trainable and how much gradient-plus-Adam state those trainable parameters imply under an fp32 proxy.

This is **not peak training memory or speed**. Every strategy still carries model weights and activations; kernels, precision, checkpointing, sequence length, batch size, and hardware determine the rest. QLoRA is omitted because no QLoRA checkpoint was trained here and its main saving is frozen-base residency, not adapter count.

> **PyTorch → Keras:** PyTorch counts trainable values with `sum(p.numel() for p in model.parameters() if p.requires_grad)`. The Keras equivalent is `sum(np.prod(weight.shape) for weight in model.trainable_weights)`. In either framework, parameter counts support an update-state estimate only; they do not measure peak memory or speed.

In [ ]:
# Compare measured trainable counts and an explicitly bounded update-state proxy.
total_params = sum(parameter.numel() for parameter in base_model.parameters())
full_ft_params = total_params
partial_ft_params = sum(
    parameter.numel() for parameter in freeze_model.parameters() if parameter.requires_grad
)
lora_params = sum(
    parameter.numel() for parameter in lora_pt_model.parameters() if parameter.requires_grad
)

techniques = ["Full fine-tuning", "Partial freezing", "LoRA (r=8)"]
param_counts = [full_ft_params, partial_ft_params, lora_params]
param_pcts = [count / total_params * 100 for count in param_counts]
colors = ["steelblue", "coral", "mediumseagreen"]

# fp32 gradient (4 bytes) + two fp32 Adam moments (8 bytes) per trainable parameter.
# Model weights, activations, temporary buffers, and allocator overhead are deliberately excluded.
update_state_bytes_per_param = 12
update_state_mb = [
    count * update_state_bytes_per_param / 1024**2 for count in param_counts
]

fig, (count_axis, state_axis) = plt.subplots(1, 2, figsize=(14, 5.5))

count_bars = count_axis.bar(techniques, param_counts, color=colors, edgecolor="black")
count_axis.set_yscale("log")
count_axis.set_ylabel("Trainable parameters (log scale)")
count_axis.set_title("Measured Trainable Parameter Count", fontweight="bold")
count_axis.grid(axis="y", alpha=0.25)
for bar, count, percent in zip(count_bars, param_counts, param_pcts):
    count_axis.annotate(
        f"{count:,}\n{percent:.2f}%",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
    )

state_bars = state_axis.barh(techniques, update_state_mb, color=colors, edgecolor="black")
state_axis.invert_yaxis()
state_axis.set_xscale("log")
state_axis.set_xlabel("Estimated fp32 gradient + Adam state (MiB, log scale)")
state_axis.set_title("Trainable-State Proxy, Not Peak Memory", fontweight="bold")
state_axis.grid(axis="x", alpha=0.25)
for bar, state_mb in zip(state_bars, update_state_mb):
    state_axis.annotate(
        f"{state_mb:,.1f} MiB",
        xy=(bar.get_width(), bar.get_y() + bar.get_height() / 2),
        xytext=(6, 0),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

print(f"{'Technique':<22} {'Trainable':>14} {'Percent':>10} {'Grad+Adam proxy':>18}")
print("-" * 68)
for technique, count, percent, state_mb in zip(
    techniques, param_counts, param_pcts, update_state_mb
):
    print(f"{technique:<22} {count:>14,} {percent:>9.2f}% {state_mb:>15,.1f} MiB")

print("\nBoundary: this proxy excludes resident weights, activations, temporary buffers, and hardware effects.")
print("Trainable-parameter ratios do not establish peak memory, elapsed time, throughput, or quality.")

### Optional Production Extension: Shrink the Selected Artifact

The training-strategy comparison is complete. Post-training quantization is a later conversion, not another fine-tuning strategy. After Riverside chooses a candidate, the next experiment applies PyTorch dynamic int8 conversion and asks:

1. How much smaller is the serialized state?
2. Does the same prompt still produce a plausible continuation?
3. Does perplexity regress on an excluded Riverside novel?

The perplexity comparison uses the same model lineage and held-out text before and after conversion; it is a regression check, not a universal quality score. Dynamic int8 represents one CPU path, not QLoRA training or a serving recommendation.

For the wider method catalog, see [Quantization in Depth](../../ai-infrastructure/06-quantization/quantization-in-depth.ipynb).

> **PyTorch → Keras:** `torch.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` converts supported linear weights to int8 for CPU inference. TensorFlow's closest equivalent uses the TFLite converter with `tf.lite.Optimize.DEFAULT`, producing a separate `.tflite` artifact rather than changing the in-memory Keras model.

In [ ]:
# Real, CPU-native post-training quantization. This is deployment conversion, not QLoRA training.
import io
import math

full_checkpoint_dir = _notebook_dir / "checkpoints" / "non-instruction-full"
config_path = full_checkpoint_dir / "config.json"
fp32_model = None
int8_model = None

if not config_path.is_file():
    print(
        "SKIP: the optional int8 experiment needs the Part 1 continued-pretraining "
        f"checkpoint at {full_checkpoint_dir}. Run Part 1 first."
    )
else:
    full_checkpoint_config = json.loads(config_path.read_text(encoding="utf-8"))
    if full_checkpoint_config.get("model_type") != base_model.config.model_type:
        raise RuntimeError(
            "The continued-pretraining checkpoint uses an incompatible architecture. "
            "Regenerate Part 1 checkpoints for MODEL_NAME."
        )
    fp32_model = AutoModelForCausalLM.from_pretrained(full_checkpoint_dir)
    fp32_model.eval()
    int8_model = torch.quantization.quantize_dynamic(
        fp32_model, {torch.nn.Linear}, dtype=torch.qint8
    )


def state_dict_size_mb(model):
    """Measure serialized parameter-state size without writing a temporary artifact."""
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / 1e6


def quick_perplexity(model, paragraphs, max_length=128):
    """Compute a same-text before/after perplexity regression check."""
    if not paragraphs:
        raise ValueError("Evaluation requires at least one paragraph.")

    model.eval()
    weighted_nll = 0.0
    predicted_tokens = 0
    with torch.inference_mode():
        for paragraph in paragraphs:
            encoded = tokenizer(
                paragraph,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            output = model(**encoded, labels=encoded["input_ids"])
            token_count = max(1, encoded["input_ids"].shape[1] - 1)
            weighted_nll += output.loss.item() * token_count
            predicted_tokens += token_count
    return math.exp(weighted_nll / predicted_tokens)


def generate_cpu(model, prompt, max_new_tokens=40):
    """Generate with CPU-resident fp32 or dynamically quantized models."""
    model.eval()
    formatted_prompt = format_instruction(prompt)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    prompt_length = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip()


if fp32_model is not None and int8_model is not None:
    fp32_mb = state_dict_size_mb(fp32_model)
    int8_mb = state_dict_size_mb(int8_model)
    print(f"fp32 state_dict size: {fp32_mb:.1f} MB")
    print(f"int8 state_dict size: {int8_mb:.1f} MB")
    print(f"Real size reduction: {(1 - int8_mb / fp32_mb) * 100:.1f}%")

    continued_pretraining_novels = {"scifi", "fantasy", "mystery"}
    held_out_novel = "historical"
    assert held_out_novel not in continued_pretraining_novels

    held_out_paragraphs = load_corpus_paragraphs(novels=[held_out_novel])
    eval_paragraph_count = 32
    sample_stride = max(1, len(held_out_paragraphs) // eval_paragraph_count)
    quant_holdout = held_out_paragraphs[::sample_stride][:eval_paragraph_count]
    print(
        f"Evaluation split: {len(quant_holdout)} paragraphs from the wholly excluded "
        f"'{held_out_novel}' novel"
    )

    fp32_ppl = quick_perplexity(fp32_model, quant_holdout)
    int8_ppl = quick_perplexity(int8_model, quant_holdout)
    print(f"\nHeld-out-novel perplexity, fp32: {fp32_ppl:.1f}")
    print(f"Held-out-novel perplexity, int8: {int8_ppl:.1f}")
    print(f"Perplexity delta from conversion: {int8_ppl - fp32_ppl:+.2f}")

    print("\n=== Same prompt, both precisions ===")
    print(f"fp32: {generate_cpu(fp32_model, PROMPT)}")
    print(f"int8: {generate_cpu(int8_model, PROMPT)}")
    print(
        "\nDynamic int8 is a deployment lever. Accept it only when size, target-hardware "
        "latency, and held-out quality changes satisfy the selected workload."
    )

---

## From Mechanisms to Candidate Behavior

The parameter story is complete: full fine-tuning changes every weight, partial freezing changes a selected slice, LoRA stores a small correction, and QLoRA additionally compresses the frozen base during adapter training.

The next prompt table is a sanity check, not a ranking. These teaching runs used different corpus slices and learning rates, so their outputs cannot isolate parameter strategy. Part 3 defines the matched study needed for that decision.

---

## Same Prompts, Four Parameter Strategies

This recap reloads the baseline, full fine-tuning, partial-freezing, and LoRA checkpoints produced across Parts 1 and 2. Each candidate receives the same raw continuation prompts with greedy decoding, and only one model is resident at a time.

Treat the table as a **visual diagnostic, not a controlled ranking**. The teaching runs used different genre slices and learning rates, so output differences combine parameter strategy with data exposure. A defensible ranking requires matched training data, seeds, token budgets, and held-out evaluation.


In [ ]:
import gc
import html
import time

from IPython.display import HTML, display
from peft import PeftModel


PARAMETER_COMPARISON_PROMPTS = {
    "Sci-fi continuation": "Aria Voss checked the Meridian's Promise status panel and",
    "Fantasy continuation": "Kerra Valmont felt all five tides simultaneously and",
    "Mystery continuation": "Elena Voss studied the 1879 survey map and realized",
}

PARAMETER_COMPARISON_CANDIDATES = [
    ("Base", "base", MODEL_NAME),
    ("Full fine-tuning", "full", "./checkpoints/non-instruction-full"),
    ("Partial freezing", "full", "./checkpoints/partial-freeze"),
    ("LoRA", "adapter", "./checkpoints/peft-lora"),
]


def load_parameter_comparison_candidate(kind, model_path):
    """Load one parameter-strategy candidate and leave the others on disk."""
    if kind == "base":
        return AutoModelForCausalLM.from_pretrained(model_path).to(device)

    artifact_path = Path(model_path)
    if not artifact_path.exists():
        raise FileNotFoundError(
            f"Missing {artifact_path}. Run the corresponding training cell before this comparison."
        )
    if kind == "full":
        return AutoModelForCausalLM.from_pretrained(artifact_path).to(device)

    adapter_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
    return PeftModel.from_pretrained(adapter_base, artifact_path).to(device)


def generate_parameter_comparison_answer(model, prompt, max_new_tokens=48):
    """Generate a deterministic raw continuation for a matched visual comparison."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}
    prompt_length = inputs["input_ids"].shape[1]
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:], skip_special_tokens=True
    ).strip() or "[model stopped immediately]"


parameter_comparison_answers = {
    prompt_label: {} for prompt_label in PARAMETER_COMPARISON_PROMPTS
}
for candidate_label, candidate_kind, candidate_path in PARAMETER_COMPARISON_CANDIDATES:
    candidate_model = load_parameter_comparison_candidate(candidate_kind, candidate_path)
    try:
        for prompt_label, prompt in PARAMETER_COMPARISON_PROMPTS.items():
            started = time.perf_counter()
            answer = generate_parameter_comparison_answer(candidate_model, prompt)
            elapsed = time.perf_counter() - started
            parameter_comparison_answers[prompt_label][candidate_label] = (
                f"{answer}\n\n[{elapsed:.1f}s]"
            )
    finally:
        del candidate_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

header = "".join(
    f"<th style='min-width:190px'>{html.escape(label)}</th>"
    for label, _, _ in PARAMETER_COMPARISON_CANDIDATES
)
body = "".join(
    "<tr>"
    f"<th style='text-align:left;vertical-align:top'>{html.escape(prompt_label)}</th>"
    + "".join(
        "<td style='vertical-align:top;white-space:pre-wrap'>"
        f"{html.escape(parameter_comparison_answers[prompt_label][label])}</td>"
        for label, _, _ in PARAMETER_COMPARISON_CANDIDATES
    )
    + "</tr>"
    for prompt_label in PARAMETER_COMPARISON_PROMPTS
)
display(
    HTML(
        "<table><thead><tr><th>Same prompt</th>"
        + header
        + "</tr></thead><tbody>"
        + body
        + "</tbody></table>"
    )
)


---

## Optional Reference: Packaging the Selected Artifact

The practical parameter-strategy lesson is complete. Continue here only if you need the filesystem boundary between a training checkpoint and a reconstructable release.

A release must pin the exact base and tokenizer, package immutable weights or adapters, record file digests, reconstruct the candidate in a fresh process, and smoke-test it before evaluation or traffic.

| Technique | What must be packaged |
| --- | --- |
| Full fine-tuning / partial freezing | Complete model weights, config, and tokenizer reference |
| LoRA | Adapter config and weights plus exact base and tokenizer revisions |
| QLoRA path | LoRA adapter plus a compatible low-bit base and runtime contract |

The disabled code below demonstrates package, verify, load, and smoke-test mechanics. It does not demonstrate production readiness: there is no independent benchmark, concurrency test, monitoring environment, or accepted rollback artifact in this notebook.

For FDE work, keep the boundary rather than the implementation details: a small adapter reduces training and storage cost, while the resident base, precision, sequence lengths, batching, and replica count still dominate inference cost.

In [ ]:
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil


@dataclass(frozen=True)
class ProductionReleaseConfig:
    """Configuration for promoting one notebook checkpoint into an immutable release."""

    release_id: str = "riverside-peft-v1"
    strategy: str = "lora"  # full, partial-freeze, lora, or qlora
    source_dir: Path = Path("./checkpoints/peft-lora")
    release_root: Path = Path("./production-artifacts")
    base_model_id: str = MODEL_NAME
    base_revision: str = "SET_EXACT_BASE_REVISION"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Return a streaming SHA-256 digest without loading a large checkpoint into RAM."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def required_artifact_groups(strategy: str) -> tuple[tuple[str, ...], ...]:
    """Return alternative required filenames for the notebook's parameter strategies."""
    if strategy in {"lora", "qlora"}:
        return (
            ("adapter_config.json",),
            ("adapter_model.safetensors", "adapter_model.bin"),
        )
    if strategy in {"full", "partial-freeze"}:
        return (
            ("config.json",),
            ("model.safetensors", "pytorch_model.bin"),
        )
    raise ValueError(f"Unsupported strategy: {strategy}")


def validate_artifact_dir(source_dir: Path, strategy: str) -> None:
    """Fail before publication when a checkpoint is incomplete or mislabeled."""
    if not source_dir.is_dir():
        raise FileNotFoundError(f"Checkpoint directory does not exist: {source_dir}")
    for alternatives in required_artifact_groups(strategy):
        if not any((source_dir / name).is_file() for name in alternatives):
            raise FileNotFoundError(
                f"Expected one of {alternatives} in {source_dir} for strategy={strategy!r}"
            )


def package_model_release(config: ProductionReleaseConfig) -> Path:
    """Copy, checksum, and atomically publish a versioned local model release."""
    if config.base_revision == "SET_EXACT_BASE_REVISION":
        raise ValueError("Pin base_revision to the exact tested model commit before publishing.")
    validate_artifact_dir(config.source_dir, config.strategy)

    config.release_root.mkdir(parents=True, exist_ok=True)
    final_dir = config.release_root / config.release_id
    staging_dir = config.release_root / f".{config.release_id}.staging"
    if final_dir.exists() or staging_dir.exists():
        raise FileExistsError(
            f"Release or staging directory already exists for {config.release_id!r}"
        )

    try:
        artifact_dir = staging_dir / "model"
        shutil.copytree(config.source_dir, artifact_dir)
        files = {
            str(path.relative_to(staging_dir)): {
                "bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in sorted(artifact_dir.rglob("*"))
            if path.is_file()
        }
        manifest = {
            "schema_version": 1,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "release": {
                **asdict(config),
                "source_dir": str(config.source_dir),
                "release_root": str(config.release_root),
            },
            "files": files,
        }
        (staging_dir / "manifest.json").write_text(
            json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8"
        )
        os.replace(staging_dir, final_dir)  # Atomic when staging and release share a filesystem.
        return final_dir
    except Exception:
        shutil.rmtree(staging_dir, ignore_errors=True)
        raise


PRODUCTION_RELEASE = ProductionReleaseConfig()
RUN_PRODUCTION_PACKAGE = False

if RUN_PRODUCTION_PACKAGE:
    published_dir = package_model_release(PRODUCTION_RELEASE)
    print(f"Published immutable release: {published_dir.resolve()}")
else:
    print("Production packaging is configured but disabled; set RUN_PRODUCTION_PACKAGE = True to publish.")


In [ ]:
from dataclasses import dataclass
import json
from pathlib import Path
import time


@dataclass(frozen=True)
class ProductionServeConfig:
    """Configuration for loading and smoke-testing one promoted release."""

    release_dir: Path = Path("./production-artifacts/riverside-peft-v1")
    device: str = device
    max_new_tokens: int = 32


def verify_release(release_dir: Path) -> dict:
    """Load the manifest and verify every published file before model loading."""
    manifest_path = release_dir / "manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for relative_path, expected in manifest["files"].items():
        artifact_path = release_dir / relative_path
        if not artifact_path.is_file():
            raise FileNotFoundError(f"Release artifact is missing: {artifact_path}")
        if artifact_path.stat().st_size != expected["bytes"]:
            raise ValueError(f"Release artifact size changed: {artifact_path}")
        if sha256_file(artifact_path) != expected["sha256"]:
            raise ValueError(f"Release artifact checksum failed: {artifact_path}")
    return manifest


def load_production_candidate(config: ProductionServeConfig):
    """Reconstruct a complete checkpoint or a PEFT adapter release for inference."""
    manifest = verify_release(config.release_dir)
    release = manifest["release"]
    strategy = release["strategy"]
    artifact_dir = config.release_dir / "model"
    base_model_id = release["base_model_id"]
    base_revision = release["base_revision"]

    serving_tokenizer = AutoTokenizer.from_pretrained(
        base_model_id, revision=base_revision
    )
    if serving_tokenizer.pad_token is None:
        serving_tokenizer.pad_token = serving_tokenizer.eos_token

    if strategy in {"lora", "qlora"}:
        if strategy == "qlora":
            if config.device != "cuda":
                raise RuntimeError("QLoRA serving requires the configured 4-bit CUDA runtime.")
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
            )
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id,
                revision=base_revision,
                quantization_config=quantization_config,
                device_map="auto",
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                base_model_id, revision=base_revision
            ).to(config.device)
        model = PeftModel.from_pretrained(base_model, artifact_dir)
    else:
        model = AutoModelForCausalLM.from_pretrained(artifact_dir).to(config.device)

    model.eval()
    return serving_tokenizer, model, manifest


def smoke_test_candidate(tokenizer, model, manifest: dict, config: ProductionServeConfig):
    """Run a deterministic probe and return observability fields without the raw prompt."""
    formatted_prompt = format_instruction(PROMPT, tokenizer)
    inputs = tokenizer(formatted_prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    inputs = {name: tensor.to(model_device) for name, tensor in inputs.items()}

    started = time.perf_counter()
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=config.max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    latency_ms = (time.perf_counter() - started) * 1000
    prompt_tokens = inputs["input_ids"].shape[1]
    output_tokens = generated.shape[1] - prompt_tokens
    completion = tokenizer.decode(
        generated[0][prompt_tokens:], skip_special_tokens=True
    ).strip()
    metrics = {
        "release_id": manifest["release"]["release_id"],
        "strategy": manifest["release"]["strategy"],
        "latency_ms": round(latency_ms, 1),
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
    }
    return completion, metrics


PRODUCTION_SERVE = ProductionServeConfig()
RUN_PRODUCTION_SMOKE_TEST = False

if RUN_PRODUCTION_SMOKE_TEST:
    production_tokenizer, production_model, production_manifest = load_production_candidate(
        PRODUCTION_SERVE
    )
    production_completion, production_metrics = smoke_test_candidate(
        production_tokenizer, production_model, production_manifest, PRODUCTION_SERVE
    )
    print(json.dumps(production_metrics, indent=2))
    print(f"Smoke-test completion: {production_completion}")
else:
    print("Production loading is configured but disabled; set RUN_PRODUCTION_SMOKE_TEST = True to run it.")


---

## Roadmap Checkpoint: From Training Choices to Model Selection

The two tuning axes are now ready to reconnect:

| Axis completed | What Riverside chose during training | What remains unknown |
| --- | --- | --- |
| Data objective (Part 1) | Continued pretraining, SFT, or DPO depending on the behavior to teach | Which behavior is required by each workload? |
| Parameter strategy (Part 2) | Full fine-tuning, partial freezing, LoRA, or the QLoRA path depending on resources | Which efficiency trade-off preserves enough quality? |
| Production artifact | Full checkpoint or versioned adapter paired with its base | Which candidate passes the relevant release evidence? |

Do not collapse those questions into one leaderboard. A model can be best at predicting Riverside prose and still be worse at following an editor's instruction.

Continue to **[Part 3: Comparison & Decision](03-llm-finetuning-comparison-and-decision.ipynb)**. Its opening roadmap shows where all seven concepts landed, then the notebook compares the measured candidates without treating objective differences as parameter-strategy results.